In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn import __version__ as sklearn_version

# --- sklearn preprocessing & pipeline ---
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator, TransformerMixin

# --- sklearn modelling & evaluation ---
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.inspection import permutation_importance
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit

# --- scipy for search distributions ---
from scipy.stats import randint

# --- TargetEncoder: requires sklearn >= 1.3 ---
sk_major, sk_minor = [int(x) for x in sklearn_version.split('.')[:2]]
if sk_major > 1 or (sk_major == 1 and sk_minor >= 3):
    from sklearn.preprocessing import TargetEncoder
    HAS_TARGET_ENCODER = True
else:
    HAS_TARGET_ENCODER = False
    print(f"WARNING: sklearn {sklearn_version} < 1.3 — TargetEncoder unavailable. Model C will be skipped.")

# --- plot style (matches project convention) ---
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 10

print(f"sklearn version : {sklearn_version}")
print(f"TargetEncoder   : {'available' if HAS_TARGET_ENCODER else 'NOT available'}")
print("All imports OK")

# Random Forest AVM — Malaysian Residential Property Valuation

## 1. Objective

This notebook trains a **Random Forest Automated Valuation Model (AVM)** to estimate Malaysian residential property transaction prices from property attributes.

**Goals:**
- Train a Random Forest regressor on cleaned transaction data (2021–2026)
- Evaluate model performance rigorously using a **chronological train/test split** (train: 2021–2025, test: 2026) to simulate deployment conditions
- Compare performance against the OLS benchmark from `linearReg2.ipynb` (R² = 0.843 on full dataset, no holdout) — noting the comparison is indicative, not identical, due to different split strategies
- Diagnose errors by segment (property type, district, price band)
- Explain the model via feature importance and local prediction analysis
- Produce an indicative valuation range based on empirical residuals

**What this notebook does NOT do:** API/backend integration, model file saving, or deployment endpoint construction. Those are future phases.

## 2. Dataset Inspection

Load the cleaned dataset and confirm actual column names, shape, target column, cardinality, and missing-value rates before committing to any feature or encoding decision.

In [ ]:
df = pd.read_excel('../processed data/Open Transaction Data Cleaned.xlsx')

print("=" * 60)
print("SHAPE")
print("=" * 60)
print(f"  Rows    : {df.shape[0]:,}")
print(f"  Columns : {df.shape[1]}")

print("\n" + "=" * 60)
print("COLUMN NAMES & DTYPES")
print("=" * 60)
print(df.dtypes.to_string())

print("\n" + "=" * 60)
print("FIRST 3 ROWS")
print("=" * 60)
df.head(3)

In [ ]:
print("=" * 60)
print("MISSING VALUES (fraction)")
print("=" * 60)
missing = df.isnull().mean().sort_values(ascending=False)
print(missing[missing > 0].to_string() if missing.max() > 0 else "  No missing values")

print("\n" + "=" * 60)
print("OBJECT COLUMN CARDINALITY")
print("=" * 60)
cat_cols = df.select_dtypes('object').columns.tolist()
for col in cat_cols:
    n_unique = df[col].nunique()
    n_blank  = (df[col].astype(str).str.strip() == '').sum()
    print(f"  {col:<25} : {n_unique:>7,} unique   |  {n_blank:>7,} blank strings")

print("\n" + "=" * 60)
print("YEAR DISTRIBUTION")
print("=" * 60)
if 'Year' in df.columns:
    print(df['Year'].value_counts().sort_index().to_string())
else:
    print("  'Year' column not found — check date parsing")

In [ ]:
# --- Target variable inspection ---
TARGET_COL = 'Price'   # expected; confirmed or corrected after running this cell

if TARGET_COL in df.columns:
    print("=" * 60)
    print(f"TARGET: '{TARGET_COL}'")
    print("=" * 60)
    desc = df[TARGET_COL].describe()
    print(desc.to_string())
    print(f"\n  Null count : {df[TARGET_COL].isnull().sum():,}")
    print(f"  Skewness   : {df[TARGET_COL].skew():.3f}")
    print(f"  Kurtosis   : {df[TARGET_COL].kurtosis():.3f}")

    # Price histogram
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(df[TARGET_COL].clip(upper=3_000_000), bins=80, color='steelblue', edgecolor='none')
    axes[0].set_title('Price Distribution (clipped at RM 3M)')
    axes[0].set_xlabel('Price (RM)')
    axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'RM {x/1e6:.1f}M'))

    axes[1].hist(np.log1p(df[TARGET_COL]), bins=80, color='darkorange', edgecolor='none')
    axes[1].set_title('log1p(Price) Distribution')
    axes[1].set_xlabel('log1p(Price)')

    plt.suptitle('Target Variable — Price', fontsize=12)
    plt.tight_layout()
    plt.show()
else:
    print(f"WARNING: '{TARGET_COL}' not found. Available columns:")
    print(df.columns.tolist())

In [ ]:
# --- Area inspection ---
print("=" * 60)
print("AREA — missing rate by Property Type")
print("=" * 60)
if 'Area' in df.columns and 'Property Type' in df.columns:
    area_miss = df.groupby('Property Type')['Area'].apply(lambda s: s.isnull().mean()).sort_values(ascending=False)
    print(area_miss.to_string())

print("\n" + "=" * 60)
print("LAND — describe")
print("=" * 60)
if 'Land' in df.columns:
    print(df['Land'].describe().to_string())

## 3. Modelling Decisions

Based on the inspection above, the following decisions are made **before writing any code**.

### 3.1 Target Transformation
`Price` has skewness = 9.786 (extreme right skew). Training will use `y = np.log1p(Price)`.
All predictions are back-transformed with `np.expm1()` to return RM values.

### 3.2 Feature Exclusions
| Feature | Decision | Reason |
|---|---|---|
| `Road Name` | **Excluded** | 121,825 unique values + 107,734 blank strings (25.9%). Infeasible for any encoding without severe overfitting. Not reliably known at inference. |
| `Scheme Name/Area` | **Excluded from baseline** | 23,718 unique values, heavy long tail. Will test in ablation (Model D) only after baseline is diagnosed. |
| `Transaction Date` | **Excluded** | Year and Month are already extracted. |

### 3.3 Area Imputation — Critical Finding
Condominiums, Flats, Low-Cost Flats, and Town Houses have **100% missing Area** in this dataset. The "Main Floor Area" field was not recorded for strata properties in the source data.
- Landed properties: Area is almost fully present (< 0.02% missing)
- Strata properties: Area is always missing → always imputed

**Implication**: Area provides no discriminating signal for strata property pricing. The model must rely on `Land`, `Property Type` dummies, and `District` for strata properties. This is documented as a data limitation.

**Imputation strategy**: `PropertyTypeMedianImputer` computes per-type median Area from the training set. For types with all-null Area (strata types), the imputed value falls back to the global training-set median. This is consistent at both training and inference time.

### 3.4 Encoding Strategy
| Feature | Encoding | Justification |
|---|---|---|
| `Property Type` (11) | OHE + handle_unknown='ignore' | Low cardinality, RF-safe |
| `Tenure` (2) | OHE + handle_unknown='ignore' | Binary categorical |
| `District` (127) | OHE + handle_unknown='ignore' | Medium cardinality; 127 OHE columns is manageable for RF |
| `Mukim` (1,343) | **Decided after Model A/B/C comparison** | High cardinality — test frequency vs target encoding |

### 3.5 Train/Test Split
Chronological: **train = `Year < 2026`** (2021–2025), **test = `Year == 2026`**. Print exact row counts after split.

### 3.6 OLS Comparison Note
The OLS benchmark (R² = 0.843) used ~304K rows (outliers removed), no train/test split, and full-dataset fitting. The RF uses a strict chronological holdout on 2026 data only. The comparison is **indicative only** — the RF test R² is not directly comparable to the OLS R² because the evaluation conditions differ.

## 4. Feature Engineering & Preprocessing Pipeline

In [ ]:
# ── Pre-split deterministic cleaning ────────────────────────────────────────
# Ensure Mukim is clean string (no NaN/None mixed with strings; TargetEncoder requires uniform type)
df['Mukim'] = df['Mukim'].fillna('Unknown').astype(str)

print(f"\nMukim NaN check: {df['Mukim'].isnull().sum()} nulls (should be 0 after fillna)")

# ── Target ───────────────────────────────────────────────────────────────────
y_all = np.log1p(df['Price'])

# ── Feature columns ──────────────────────────────────────────────────────────
FEATURES_A  = ['Property Type', 'Tenure', 'District', 'Land', 'Area', 'Year', 'Month']
FEATURES_BC = FEATURES_A + ['Mukim']

X_all_A  = df[FEATURES_A].copy()
X_all_BC = df[FEATURES_BC].copy()

print(f"\nFeatures (Model A)  : {len(FEATURES_A)}")
print(f"Features (Model B/C): {len(FEATURES_BC)}")


# ── Custom Area imputer ───────────────────────────────────────────────────────
class PropertyTypeMedianImputer(BaseEstimator, TransformerMixin):
    """
    Imputes 'Area' using per-Property-Type median from training data.
    Input X: 2-column array [Area, Property Type].
    Strata types with all-null Area fall back to global training median.
    """
    def fit(self, X, y=None):
        arr = np.array(X, dtype=object)
        area_s = pd.Series(pd.to_numeric(arr[:, 0], errors='coerce'))
        type_s = pd.Series(arr[:, 1].astype(str))
        tmp = pd.DataFrame({'Area': area_s, 'PropType': type_s})
        self.type_medians_ = tmp.groupby('PropType')['Area'].median().to_dict()
        self.global_median_ = float(area_s.median())
        return self

    def transform(self, X):
        arr  = np.array(X, dtype=object)
        area = pd.Series(pd.to_numeric(arr[:, 0], errors='coerce'))
        prop_types = pd.Series(arr[:, 1].astype(str))
        missing_mask = area.isna()
        if missing_mask.any():
            type_fill = prop_types[missing_mask].map(self.type_medians_)
            type_fill = type_fill.fillna(self.global_median_)
            area.loc[missing_mask] = type_fill.values
        return area.values.reshape(-1, 1).astype(float)

print("\nPropertyTypeMedianImputer defined.")


# ── Pipeline builder ──────────────────────────────────────────────────────────
def build_pipeline(features, mukim_encoder=None, n_estimators=100, **rf_kwargs):
    """
    mukim_encoder: None (Model A), 'frequency' (Model B), 'target' (Model C).
    OOB score enabled only for Model A (no target encoder).
    """
    land_pipe = Pipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('log',    FunctionTransformer(np.log1p))
    ])
    area_pipe = Pipeline([
        ('impute_typed', PropertyTypeMedianImputer()),
        ('log',          FunctionTransformer(np.log1p))
    ])
    pass_pipe = Pipeline([('impute', SimpleImputer(strategy='median'))])
    low_ohe   = Pipeline([('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])
    dist_ohe  = Pipeline([('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])

    transformers = [
        ('low_ohe',  low_ohe,   ['Property Type', 'Tenure']),
        ('dist_ohe', dist_ohe,  ['District']),
        ('land',     land_pipe, ['Land']),
        ('area',     area_pipe, ['Area', 'Property Type']),
        ('passnum',  pass_pipe, ['Year', 'Month']),
    ]

    if mukim_encoder == 'frequency':
        class FrequencyEncoder(BaseEstimator, TransformerMixin):
            def fit(self, X, y=None):
                vals = pd.Series(np.array(X).ravel().astype(str))
                self.count_map_ = vals.value_counts().to_dict()
                return self
            def transform(self, X):
                vals = pd.Series(np.array(X).ravel().astype(str))
                return vals.map(self.count_map_).fillna(1).values.reshape(-1, 1).astype(float)
        transformers.append(('mukim_freq', FrequencyEncoder(), ['Mukim']))

    elif mukim_encoder == 'target' and HAS_TARGET_ENCODER:
        # smooth='auto' handles regularisation; no explicit cv to avoid Python 3.14 issues
        transformers.append((
            'mukim_te',
            TargetEncoder(target_type='continuous', smooth='auto'),
            ['Mukim']
        ))

    ct  = ColumnTransformer(transformers, remainder='drop')
    oob = (mukim_encoder is None)
    rf_params = dict(n_estimators=n_estimators, max_features=1/3,
                     oob_score=oob, random_state=42, n_jobs=-1)
    rf_params.update(rf_kwargs)
    return Pipeline([('prep', ct), ('rf', RandomForestRegressor(**rf_params))])


print("build_pipeline() helper defined.")

## 5. Chronological Train/Test Split

Split on `Year`: **train = 2021–2025**, **test = 2026**.

Property markets are non-stationary. Evaluating on the most recent available year (2026) simulates real deployment conditions: the model is trained on all historical transactions and tested on data it has never seen in strict time order. This prevents future price levels from leaking into training.

In [ ]:
# Chronological split: train on 2021–2025, test on 2026.
train_mask = df['Year'] < 2026
test_mask  = df['Year'] == 2026

X_train_A  = X_all_A[train_mask].reset_index(drop=True)
X_test_A   = X_all_A[test_mask].reset_index(drop=True)
X_train_BC = X_all_BC[train_mask].reset_index(drop=True)
X_test_BC  = X_all_BC[test_mask].reset_index(drop=True)
y_train    = y_all[train_mask].reset_index(drop=True)
y_test     = y_all[test_mask].reset_index(drop=True)

# Keep test original indices for segment analysis
test_orig_idx = df.index[test_mask].tolist()

print("=" * 50)
print("CHRONOLOGICAL TRAIN/TEST SPLIT")
print("=" * 50)
print(f"  Train  : {len(X_train_A):,} rows  (Year 2021–2025)")
print(f"  Test   : {len(X_test_A):,} rows  (Year 2026)")
print(f"  Test % : {len(X_test_A)/len(X_all_A)*100:.1f}%")
print()
print("Test set breakdown:")
print(df.loc[test_mask, 'Month'].value_counts().sort_index()
      .rename('Count').rename_axis('Month (2026)').to_string())


## 6. Baseline Random Forest — Model A

Model A uses: `Property Type`, `Tenure`, `District`, `Land`, `Area`, `Year`, `Month`.
No Mukim — its encoding is decided separately after comparing Models A/B/C.

Parameters: 100 trees, `max_features=1/3` (Breiman's regression convention), `oob_score=True`.

In [ ]:
pipeline_A = build_pipeline(FEATURES_A, mukim_encoder=None, n_estimators=50)
pipeline_A.fit(X_train_A, y_train)

oob_r2 = pipeline_A.named_steps['rf'].oob_score_
print(f"OOB R² (training-set estimate, Model A, 50 trees): {oob_r2:.4f}")
print("(50 trees for speed at baseline; final model will use more trees after tuning)")

In [ ]:
def evaluate_model(pipeline, X_tr, y_tr, X_te, y_te, label=''):
    """
    Evaluate a fitted pipeline. Returns a metrics dict.
    All RM-space metrics computed after expm1 back-transform.
    R² reported in both log-space (for OLS comparison) and RM-space.
    """
    pred_tr_log = pipeline.predict(X_tr)
    pred_te_log = pipeline.predict(X_te)

    # back-transform
    y_tr_rm   = np.expm1(y_tr)
    y_te_rm   = np.expm1(y_te)
    pred_tr_rm = np.expm1(pred_tr_log)
    pred_te_rm = np.expm1(pred_te_log)

    def metrics(y_true_log, y_pred_log, y_true_rm, y_pred_rm, split):
        r2_log = r2_score(y_true_log, y_pred_log)
        r2_rm  = r2_score(y_true_rm, y_pred_rm)
        rmse   = np.sqrt(mean_squared_error(y_true_rm, y_pred_rm))
        mae    = mean_absolute_error(y_true_rm, y_pred_rm)
        medae  = float(np.median(np.abs(y_true_rm - y_pred_rm)))
        med_price = float(np.median(y_true_rm))
        rmse_pct = rmse / med_price * 100
        return dict(split=split, r2_log=r2_log, r2_rm=r2_rm,
                    rmse=rmse, mae=mae, medae=medae,
                    med_price=med_price, rmse_pct=rmse_pct)

    tr = metrics(y_tr,  pred_tr_log, y_tr_rm,  pred_tr_rm,  'Train')
    te = metrics(y_te,  pred_te_log, y_te_rm,  pred_te_rm,  'Test')

    print(f"\n{'='*60}")
    print(f"  {label}")
    print(f"{'='*60}")
    header = f"{'Metric':<22} {'Train':>12} {'Test':>12}"
    print(header)
    print("-" * len(header))
    for k, fmt in [('r2_log',   '.4f'), ('r2_rm',    '.4f'),
                   ('rmse',     ',.0f'), ('mae',      ',.0f'),
                   ('medae',    ',.0f'), ('rmse_pct', '.1f')]:
        label_k = k.replace('_', ' ').upper()
        tv = format(tr[k], fmt)
        ev = format(te[k], fmt)
        unit = '' if 'r2' in k or 'pct' in k else 'RM '
        pct  = '%' if 'pct' in k else ''
        print(f"  {label_k:<20} {unit+tv+pct:>12} {unit+ev+pct:>12}")

    return tr, te, pred_te_log


tr_A, te_A, pred_te_A = evaluate_model(pipeline_A, X_train_A, y_train,
                                        X_test_A,  y_test,
                                        label='MODEL A — Baseline RF (no Mukim)')

In [ ]:
def plot_evaluation(y_test_log, pred_test_log, label='Model', df_test_meta=None):
    """
    5-panel evaluation plot suite. All inline, no file saving.
    df_test_meta: DataFrame with ['Year','Month'] for temporal drift plot.
    """
    y_rm    = np.expm1(y_test_log)
    p_rm    = np.expm1(pred_test_log)
    resid   = np.array(y_test_log) - pred_test_log
    clip_max = 5_000_000

    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    axes = axes.ravel()

    # 1. Actual vs Predicted
    ax = axes[0]
    clip_mask = (y_rm <= clip_max) & (p_rm <= clip_max)
    ax.scatter(y_rm[clip_mask]/1e6, p_rm[clip_mask]/1e6,
               alpha=0.15, s=4, color='steelblue', rasterized=True)
    lim = max(y_rm[clip_mask].max(), p_rm[clip_mask].max()) / 1e6 * 1.05
    ax.plot([0, lim], [0, lim], 'r--', lw=1.2, label='Perfect')
    ax.set_xlabel('Actual (RM M)')
    ax.set_ylabel('Predicted (RM M)')
    ax.set_title('Actual vs Predicted (test, clipped RM 5M)')
    ax.legend(fontsize=8)

    # 2. Residuals vs Predicted (log-space)
    ax = axes[1]
    ax.scatter(pred_test_log, resid, alpha=0.1, s=3, color='darkorange', rasterized=True)
    ax.axhline(0, color='black', lw=1)
    ax.set_xlabel('Predicted log1p(Price)')
    ax.set_ylabel('Residual (actual − predicted, log)')
    ax.set_title('Residuals vs Predicted (log-space)')

    # 3. Residual distribution
    ax = axes[2]
    ax.hist(resid, bins=80, color='teal', edgecolor='none', density=True, alpha=0.7)
    from scipy.stats import norm
    mu, sigma = resid.mean(), resid.std()
    xs = np.linspace(resid.min(), resid.max(), 300)
    ax.plot(xs, norm.pdf(xs, mu, sigma), 'r-', lw=1.5, label=f'N({mu:.3f},{sigma:.3f})')
    ax.axvline(0, color='black', lw=1)
    ax.axvline(mu - 1.96*sigma, color='orange', lw=1, ls='--')
    ax.axvline(mu + 1.96*sigma, color='orange', lw=1, ls='--', label='±1.96σ')
    ax.set_xlabel('Residual (log-space)')
    ax.set_title('Residual Distribution')
    ax.legend(fontsize=8)

    # 4. Temporal drift: median residual by month
    ax = axes[3]
    if df_test_meta is not None:
        tmp = df_test_meta.copy()
        tmp['resid'] = resid
        tmp['period'] = tmp['Year'].astype(str) + '-' + tmp['Month'].astype(str).str.zfill(2)
        drift = tmp.groupby('period')['resid'].median().sort_index()
        ax.bar(range(len(drift)), drift.values, color='steelblue', alpha=0.8)
        ax.set_xticks(range(len(drift)))
        ax.set_xticklabels(drift.index, rotation=60, ha='right', fontsize=7)
        ax.axhline(0, color='black', lw=1)
        ax.set_title('Median Residual by Month (temporal drift)')
        ax.set_ylabel('Median residual (log)')
    else:
        ax.axis('off')

    # 5. RMSE by price band
    ax = axes[4]
    bands = [(0, 300_000, '≤300K'), (300_000, 500_000, '300K–500K'),
             (500_000, 1_000_000, '500K–1M'), (1_000_000, np.inf, '>1M')]
    band_rmse, band_n = [], []
    for lo, hi, _ in bands:
        m = (y_rm >= lo) & (y_rm < hi)
        if m.sum() > 0:
            band_rmse.append(np.sqrt(mean_squared_error(y_rm[m], p_rm[m])))
            band_n.append(m.sum())
        else:
            band_rmse.append(0); band_n.append(0)
    bars = ax.bar([b[2] for b in bands], [r/1e3 for r in band_rmse],
                  color=['#4CAF50','#2196F3','#FF9800','#F44336'])
    for bar, n in zip(bars, band_n):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f'n={n:,}', ha='center', va='bottom', fontsize=7)
    ax.set_ylabel('RMSE (RM 000)')
    ax.set_title('RMSE by Price Band')

    axes[5].axis('off')   # spare panel

    plt.suptitle(f'{label} — Evaluation Plots (Test Set)', fontsize=12)
    plt.tight_layout()
    plt.show()


# Test metadata for temporal drift plot
df_test_meta = df.loc[test_orig_idx, ['Year','Month']].reset_index(drop=True)

plot_evaluation(y_test, pred_te_A, label='Model A (baseline)',
                df_test_meta=df_test_meta)

## 7. Baseline Diagnosis

See the evaluation table and plots above for actual metric values from Model A (50 trees, no Mukim, 2026 test set).

### Finding 1 — Overfitting
Train R² is substantially higher than test R². The gap reflects RF over-fitting to individual training samples (unlimited depth, min_samples_leaf=1) AND the harder out-of-time evaluation — 2026 prices may have shifted relative to the 2021–2025 training distribution. Tuning targets `max_depth` and `min_samples_leaf`.

### Finding 2 — RMSE vs Median AE
Headline RMSE is dominated by errors on luxury properties (>RM 2M). Median Absolute Error is the more honest AVM accuracy measure for the core market (RM 200K–600K). RMSE as % of median price is the primary tuning target.

### Finding 3 — OLS comparison (indicative only)
OLS R² = 0.843 was computed on the full dataset with no holdout and ~304K rows after outlier removal. RF test R² is computed on the 2026 held-out set only. The comparison is indicative: OLS had no holdout and used different rows.

### Finding 4 — Mukim not yet included
Model A uses District (127 OHE) but not Mukim (1,343 unique). Models B and C test whether adding Mukim sub-district signal improves performance on the 2026 test set.

### Tuning strategy
- **Priority 1**: Reduce `max_depth` (currently unlimited). Try depth 10–25.
- **Priority 2**: Increase `min_samples_leaf` (1 → 2–10) to regularise leaf nodes.
- **Priority 3**: After Mukim comparison, tune the best variant on an 80K chronological sample with 5-fold TimeSeriesSplit CV.

## 8. Mukim Encoding Comparison — Models A / B / C

Compare three variants to decide whether including Mukim improves test performance:
- **Model A**: District only (already trained above)
- **Model B**: District + Mukim (frequency encoding — Mukim replaced by its count in training set)
- **Model C**: District + Mukim (target encoding with CV=5 — Mukim replaced by its training-set mean log-price, blended toward global mean)

Decision criterion: choose the model with the highest test R² (log-space) and no obvious train/test gap widening.

Note: OOB R² is **not** reported for Models B and C — target encoding inside the pipeline introduces subtle leakage into the OOB bootstrap mechanism. Rely on chronological test R² only.

In [ ]:
# ── Model B: Mukim + frequency encoding ──────────────────────────────────────
print("Training Model B (Mukim + frequency encoding, 50 trees)...")
pipeline_B = build_pipeline(FEATURES_BC, mukim_encoder='frequency', n_estimators=50)
pipeline_B.fit(X_train_BC, y_train)
tr_B, te_B, pred_te_B = evaluate_model(pipeline_B, X_train_BC, y_train,
                                        X_test_BC, y_test,
                                        label='MODEL B — RF + Mukim (frequency encoding)')

# ── Model C: Mukim + target encoding ─────────────────────────────────────────
if HAS_TARGET_ENCODER:
    print("\nTraining Model C (Mukim + target encoding, 50 trees)...")
    pipeline_C = build_pipeline(FEATURES_BC, mukim_encoder='target', n_estimators=50)
    pipeline_C.fit(X_train_BC, y_train)
    tr_C, te_C, pred_te_C = evaluate_model(pipeline_C, X_train_BC, y_train,
                                            X_test_BC, y_test,
                                            label='MODEL C — RF + Mukim (target encoding)')
else:
    print("Model C skipped — sklearn < 1.3")
    tr_C = te_C = pred_te_C = None

In [ ]:
# ── Comparison table ─────────────────────────────────────────────────────────
print("\n" + "=" * 80)
print("MUKIM ENCODING COMPARISON")
print("=" * 80)
rows = [
    ('OLS benchmark', '—', '0.8430', '—', '—', '—', '—', 'No holdout, 304K rows'),
    ('Model A (no Mukim)',
     f"{tr_A['r2_log']:.4f}", f"{te_A['r2_log']:.4f}",
     f"RM {te_A['rmse']:,.0f}", f"RM {te_A['mae']:,.0f}",
     f"RM {te_A['medae']:,.0f}", f"{te_A['rmse_pct']:.1f}%",
     '100 trees, District OHE only'),
    ('Model B (freq Mukim)',
     f"{tr_B['r2_log']:.4f}", f"{te_B['r2_log']:.4f}",
     f"RM {te_B['rmse']:,.0f}", f"RM {te_B['mae']:,.0f}",
     f"RM {te_B['medae']:,.0f}", f"{te_B['rmse_pct']:.1f}%",
     '100 trees, Mukim freq enc'),
]
if te_C:
    rows.append((
        'Model C (target Mukim)',
        f"{tr_C['r2_log']:.4f}", f"{te_C['r2_log']:.4f}",
        f"RM {te_C['rmse']:,.0f}", f"RM {te_C['mae']:,.0f}",
        f"RM {te_C['medae']:,.0f}", f"{te_C['rmse_pct']:.1f}%",
        '100 trees, Mukim TargetEnc'
    ))

hdr = f"{'Model':<26} {'Tr R²':>7} {'Te R²':>7} {'RMSE':>12} {'MAE':>11} {'MedAE':>11} {'RMSE%':>7} Notes"
print(hdr)
print("-" * len(hdr))
for r in rows:
    print(f"{r[0]:<26} {r[1]:>7} {r[2]:>7} {r[3]:>12} {r[4]:>11} {r[5]:>11} {r[6]:>7}  {r[7]}")

# Choose best model based on test R² (log)
best_te = max([(te_A, 'A', pipeline_A, X_test_A, FEATURES_A)],
              key=lambda x: x[0]['r2_log'])
if te_B['r2_log'] > best_te[0]['r2_log']:
    best_te = (te_B, 'B', pipeline_B, X_test_BC, FEATURES_BC)
if te_C and te_C['r2_log'] > best_te[0]['r2_log']:
    best_te = (te_C, 'C', pipeline_C, X_test_BC, FEATURES_BC)

_, best_model_label, best_pipeline, best_X_test, best_features = best_te
pred_te_best = best_pipeline.predict(best_X_test)
print(f"\n>>> Best model for further analysis: Model {best_model_label} (Test R² = {best_te[0]['r2_log']:.4f})")

## 9. Segment-Level Error Analysis

Analyse test-set errors by: Property Type, Tenure, Price Band, and top 15 Districts.

In [ ]:
seg = df.loc[test_orig_idx, ['Property Type', 'District', 'Tenure', 'Price']].reset_index(drop=True).copy()
seg['y_true_rm']    = np.expm1(y_test.values)
seg['y_pred_rm']    = np.expm1(pred_te_best)
seg['abs_error']    = np.abs(seg['y_true_rm'] - seg['y_pred_rm'])
seg['pct_error']    = seg['abs_error'] / seg['y_true_rm'] * 100
seg['price_band']   = pd.cut(seg['y_true_rm'],
                              bins=[0, 300_000, 500_000, 1_000_000, np.inf],
                              labels=['≤300K', '300K–500K', '500K–1M', '>1M'])

def seg_table(group_col, data=seg, top_n=None):
    grp = data.groupby(group_col)
    tbl = grp.apply(lambda g: pd.Series({
        'N': len(g),
        'Median Price (RM)': int(g['y_true_rm'].median()),
        'RMSE (RM)': int(np.sqrt(mean_squared_error(g['y_true_rm'], g['y_pred_rm']))),
        'MAE (RM)':  int(mean_absolute_error(g['y_true_rm'], g['y_pred_rm'])),
        'Median % Error': round(g['pct_error'].median(), 1)
    }), include_groups=False).sort_values('Median % Error', ascending=False)
    if top_n:
        top_dists = data[group_col].value_counts().head(top_n).index
        tbl = tbl.loc[tbl.index.isin(top_dists)]
        tbl = tbl.sort_values('Median % Error', ascending=False)
    return tbl

print("=== BY PROPERTY TYPE ===")
print(seg_table('Property Type').to_string())

print("\n=== BY TENURE ===")
print(seg_table('Tenure').to_string())

print("\n=== BY PRICE BAND ===")
print(seg_table('price_band').to_string())

print("\n=== TOP 15 DISTRICTS (by test volume) ===")
print(seg_table('District', top_n=15).to_string())

In [ ]:
# Heatmap: Median % Error by Property Type × Price Band
pivot = seg.pivot_table(values='pct_error', index='Property Type',
                         columns='price_band', aggfunc='median')
fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='RdYlGn_r',
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Median % Error'})
ax.set_title(f'Median % Error — Property Type × Price Band (Model {best_model_label} test set)')
ax.set_xlabel('Price Band')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

## 10. Hyperparameter Tuning

**Strategy**: Diagnosis showed overfitting (train R² >> test R²). Main regularisation levers: `max_depth`, `min_samples_leaf`.

To keep tuning practical on a ~411K row training set:
1. Sample 80K rows proportionally by Year (preserving chronological order within the sample)
2. Run `RandomizedSearchCV` with `TimeSeriesSplit(n_splits=5)` on the sorted sample — row order must be preserved so folds respect time
3. Identify best parameters → refit on the **full training set**
4. Evaluate the refitted model on the 2026 test set

In [ ]:
# ── Prepare tuning features (use best model's feature set) ───────────────────
if best_model_label == 'A':
    X_train_tune = X_train_A
    mukim_enc_for_tune = None
else:
    X_train_tune = X_train_BC
    mukim_enc_for_tune = 'frequency' if best_model_label == 'B' else 'target'

# Chronologically ordered sample: proportional to Year distribution, sorted by Year/Month.
# TimeSeriesSplit works on row position — chronological order must be preserved.
SAMPLE_SIZE = 80_000
train_df_for_sample = X_train_tune.copy()
train_df_for_sample['__y']     = y_train.values
train_df_for_sample['__Year']  = df.loc[train_mask, 'Year'].values
train_df_for_sample['__Month'] = df.loc[train_mask, 'Month'].values

total_rows = len(train_df_for_sample)
pieces = []
for yr in sorted(train_df_for_sample['__Year'].unique()):
    grp  = train_df_for_sample[train_df_for_sample['__Year'] == yr]
    n_yr = max(1, round(len(grp) / total_rows * SAMPLE_SIZE))
    n_yr = min(n_yr, len(grp))
    pieces.append(grp.sample(n=n_yr, random_state=42))

tune_sample = (pd.concat(pieces)
               .sort_values(['__Year', '__Month'])
               .reset_index(drop=True))

X_tune = tune_sample[X_train_tune.columns]
y_tune = tune_sample['__y']

print(f"Tuning sample : {len(X_tune):,} rows (chronological, proportional by year)")
print(f"Year range    : {tune_sample['__Year'].min()} – {tune_sample['__Year'].max()}")
print(f"Year counts   :")
print(tune_sample['__Year'].value_counts().sort_index().to_string())


In [ ]:
param_dist = {
    'rf__max_depth':         [10, 15, 20, 25, None],
    'rf__min_samples_leaf':  [1, 2, 5, 10],
    'rf__min_samples_split': [2, 5, 10],
    'rf__max_features':      [0.2, 0.33, 0.5, 'sqrt'],
    'rf__n_estimators':      [100, 200],
}

tscv = TimeSeriesSplit(n_splits=5)

search_pipeline = build_pipeline(list(X_tune.columns),
                                  mukim_encoder=mukim_enc_for_tune,
                                  n_estimators=100)

rscv = RandomizedSearchCV(
    estimator=search_pipeline,
    param_distributions=param_dist,
    n_iter=15,
    scoring='neg_root_mean_squared_error',
    cv=tscv,
    n_jobs=-1,
    random_state=42,
    verbose=1,
    refit=False
)

print(f"Running RandomizedSearchCV on {len(X_tune):,} rows, 15 iterations, 5-fold TimeSeriesSplit...")
rscv.fit(X_tune, y_tune)

best_params = rscv.best_params_
best_cv_rmse = -rscv.best_score_
print(f"\nBest CV RMSE (log-space): {best_cv_rmse:.4f}")
print("Best parameters:")
for k, v in sorted(best_params.items()):
    print(f"  {k:<35}: {v}")


In [ ]:
# ── Refit best params on full training set ────────────────────────────────────
rf_tuned_params = {k.replace('rf__', ''): v for k, v in best_params.items()
                   if k.startswith('rf__')}

pipeline_tuned = build_pipeline(
    list(X_train_tune.columns),
    mukim_encoder=mukim_enc_for_tune,
    **rf_tuned_params
)
print("Refitting tuned model on full training set...")
pipeline_tuned.fit(X_train_tune, y_train)
print("Done.")

if best_model_label == 'A':
    X_test_tuned = X_test_A
else:
    X_test_tuned = X_test_BC

tr_tuned, te_tuned, pred_te_tuned = evaluate_model(
    pipeline_tuned, X_train_tune, y_train,
    X_test_tuned, y_test,
    label=f'TUNED MODEL ({best_model_label}) — {rf_tuned_params}'
)

In [ ]:
# Final model comparison table
print("\n" + "=" * 95)
print("FINAL MODEL COMPARISON")
print("=" * 95)
hdr = f"{'Model':<30} {'Tr R²':>7} {'Te R²':>7} {'RMSE':>12} {'MAE':>11} {'MedAE':>11} {'RMSE%':>7}  Notes"
print(hdr)
print("-" * len(hdr))

summary_rows = [
    ('OLS benchmark', '—', '0.8430', '—', '—', '—', '—', 'Full dataset, no holdout'),
    (f'Model A baseline',
     f"{tr_A['r2_log']:.4f}", f"{te_A['r2_log']:.4f}",
     f"RM {te_A['rmse']:,.0f}", f"RM {te_A['mae']:,.0f}",
     f"RM {te_A['medae']:,.0f}", f"{te_A['rmse_pct']:.1f}%",
     '100 trees, default depth'),
    (f'Best pre-tuning (Model {best_model_label})',
     f"{best_te[0]['r2_log']:.4f}", f"{best_te[0]['r2_log']:.4f}",
     f"RM {best_te[0]['rmse']:,.0f}", f"RM {best_te[0]['mae']:,.0f}",
     f"RM {best_te[0]['medae']:,.0f}", f"{best_te[0]['rmse_pct']:.1f}%",
     'Before tuning'),
    (f'Tuned (Model {best_model_label})',
     f"{tr_tuned['r2_log']:.4f}", f"{te_tuned['r2_log']:.4f}",
     f"RM {te_tuned['rmse']:,.0f}", f"RM {te_tuned['mae']:,.0f}",
     f"RM {te_tuned['medae']:,.0f}", f"{te_tuned['rmse_pct']:.1f}%",
     str(rf_tuned_params)),
]
for r in summary_rows:
    print(f"{r[0]:<30} {r[1]:>7} {r[2]:>7} {r[3]:>12} {r[4]:>11} {r[5]:>11} {r[6]:>7}  {r[7]}")

# Update best pipeline to tuned version
final_pipeline = pipeline_tuned
final_X_test   = X_test_tuned
final_pred_te  = pred_te_tuned
final_features = list(X_train_tune.columns)
print(f"\n>>> Final model: Tuned Model {best_model_label}")

## 11. Feature Importance

Three levels: (1) RF built-in impurity importance, (2) permutation importance on a test sample, (3) SHAP if runtime allows.

In [ ]:
# ── 11.1 RF built-in impurity importance ─────────────────────────────────────
rf_model = final_pipeline.named_steps['rf']
prep     = final_pipeline.named_steps['prep']

# Build feature names without calling prep.get_feature_names_out() — custom
# transformers (PropertyTypeMedianImputer, FrequencyEncoder) do not implement
# get_feature_names_out, which breaks the ColumnTransformer chain in sklearn 1.8+.
def _build_feat_names(prep):
    names = []
    for tname, trans, cols in prep.transformers_:
        if trans == 'drop':
            continue
        if tname in ('low_ohe', 'dist_ohe'):
            raw = trans.named_steps['ohe'].get_feature_names_out(cols)
            names.extend([f'{tname}__{n}' for n in raw])
        elif tname == 'land':
            names.append('land__log1p_Land')
        elif tname == 'area':
            names.append('area__log1p_Area')
        elif tname == 'passnum':
            names.extend([f'passnum__{c}' for c in cols])
        elif tname in ('mukim_freq', 'mukim_te'):
            names.append(f'{tname}__Mukim')
    return np.array(names)

feat_names  = _build_feat_names(prep)
importances = rf_model.feature_importances_

assert len(feat_names) == len(importances), (
    f"Feature name count ({len(feat_names)}) != importances count ({len(importances)}). "
    "Check _build_feat_names mirrors the ColumnTransformer output exactly."
)

imp_df = pd.DataFrame({'feature': feat_names, 'importance': importances})
imp_df = imp_df.sort_values('importance', ascending=False).reset_index(drop=True)

# Aggregate back to original feature groups
group_patterns = {
    'Property Type': 'low_ohe__Property Type',
    'Tenure':        'low_ohe__Tenure',
    'District':      'dist_ohe__District',
    'Mukim':         'mukim_',
    'Land':          'land__',
    'Area':          'area__',
    'Year':          'passnum__Year',
    'Month':         'passnum__Month',
}

group_imp = {}
for group, pattern in group_patterns.items():
    mask = imp_df['feature'].str.startswith(pattern)
    if mask.any():
        group_imp[group] = imp_df.loc[mask, 'importance'].sum()

group_imp_df = pd.DataFrame(list(group_imp.items()), columns=['Feature Group', 'Importance'])
group_imp_df = group_imp_df.sort_values('Importance', ascending=False).reset_index(drop=True)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

top30 = imp_df.head(30)
axes[0].barh(top30['feature'][::-1], top30['importance'][::-1], color='steelblue')
axes[0].set_title('Top 30 Features — RF Impurity Importance')
axes[0].set_xlabel('Importance')
axes[0].tick_params(axis='y', labelsize=7)

axes[1].barh(group_imp_df['Feature Group'][::-1],
             group_imp_df['Importance'][::-1], color='darkorange')
axes[1].set_title('Feature Group Importance (aggregated)')
axes[1].set_xlabel('Total Importance')

plt.tight_layout()
plt.show()

print("\nFeature Group Importance (ranked):")
print(group_imp_df.to_string(index=False))

In [ ]:
# ── 11.2 Permutation importance (on 5K test sample) ──────────────────────────
# permutation_importance on the full Pipeline shuffles *input* columns of X_perm
# (one importance per input feature), not OHE-expanded columns.
import time

perm_sample_size = min(5000, len(final_X_test))
perm_idx = np.random.RandomState(42).choice(len(final_X_test), perm_sample_size, replace=False)
X_perm = final_X_test.iloc[perm_idx]
y_perm = y_test.iloc[perm_idx]

print(f"Running permutation importance on {perm_sample_size} test samples...")
t0 = time.time()
perm_result = permutation_importance(
    final_pipeline, X_perm, y_perm,
    n_repeats=10, random_state=42, n_jobs=-1, scoring='r2'
)
print(f"Done in {time.time()-t0:.1f}s")

# Feature names = input column names (one per input feature, before preprocessing)
perm_feat_names = list(X_perm.columns)
perm_df = pd.DataFrame({
    'feature':    perm_feat_names,
    'perm_mean':  perm_result.importances_mean,
    'perm_std':   perm_result.importances_std
}).sort_values('perm_mean', ascending=False).reset_index(drop=True)

# Each input feature IS its own group — no aggregation needed
perm_group_df = (perm_df[['feature', 'perm_mean']]
                 .rename(columns={'feature': 'Feature Group', 'perm_mean': 'Perm Importance'})
                 .sort_values('Perm Importance', ascending=False)
                 .reset_index(drop=True))

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

axes[0].barh(perm_df['feature'][::-1], perm_df['perm_mean'][::-1],
             xerr=perm_df['perm_std'][::-1], color='teal', alpha=0.8, capsize=3)
axes[0].set_title('Input Features — Permutation Importance (mean ± std, R² drop)')
axes[0].set_xlabel('Mean R² decrease')
axes[0].tick_params(axis='y', labelsize=8)

axes[1].barh(perm_group_df['Feature Group'][::-1],
             perm_group_df['Perm Importance'][::-1], color='purple', alpha=0.8)
axes[1].set_title('Features — Permutation Importance (sorted)')
axes[1].set_xlabel('Mean R² decrease')

plt.tight_layout()
plt.show()

print("\nComparison: Impurity (grouped) vs Permutation (input feature) importance:")
merged = (group_imp_df
          .rename(columns={'Feature Group': 'Feature', 'Importance': 'Impurity Imp.'})
          .merge(
              perm_df[['feature', 'perm_mean']].rename(
                  columns={'feature': 'Feature', 'perm_mean': 'Perm Imp.'}),
              on='Feature', how='outer'
          )
          .fillna(0)
          .sort_values('Impurity Imp.', ascending=False))
print(merged.to_string(index=False))

In [ ]:
# ── 11.3 SHAP (optional — skip if too slow) ───────────────────────────────────
try:
    import shap
    import time

    SHAP_SAMPLE = 3000
    shap_idx  = np.random.RandomState(42).choice(len(final_X_test), SHAP_SAMPLE, replace=False)
    X_shap_df = final_X_test.iloc[shap_idx]
    X_shap    = prep.transform(X_shap_df)

    print(f"Running SHAP TreeExplainer on {SHAP_SAMPLE} samples...")
    t0 = time.time()
    explainer   = shap.TreeExplainer(rf_model)
    shap_values = explainer.shap_values(X_shap)
    print(f"Done in {time.time()-t0:.1f}s")

    # Global SHAP bar plot
    shap.summary_plot(shap_values, X_shap, feature_names=feat_names,
                      plot_type='bar', max_display=20, show=False)
    plt.title('SHAP — Global Feature Importance (mean |SHAP|)')
    plt.tight_layout()
    plt.show()

    # Beeswarm
    shap.summary_plot(shap_values, X_shap, feature_names=feat_names,
                      max_display=15, show=False)
    plt.title('SHAP — Beeswarm (feature value vs impact)')
    plt.tight_layout()
    plt.show()

    HAS_SHAP = True
    print("SHAP completed successfully.")

except ImportError:
    print("shap not installed — skipping. Install with: pip install shap")
    HAS_SHAP = False
    explainer   = None
    shap_values = None
except Exception as e:
    print(f"SHAP failed: {e}")
    HAS_SHAP = False
    explainer   = None
    shap_values = None

## 12. Local Prediction Explanation

Apply the final model to a representative property and explain what drives the prediction.

In [ ]:
# Representative property: 2-Storey Terraced, Johor Bahru, Freehold, 150 sqm land, 110 sqm area
sample_input = {
    'Property Type': '2 - 2 1/2 Storey Terraced',
    'Tenure':        'Freehold',
    'District':      'Johor Bahru',
    'Land':          150.0,
    'Area':          110.0,
    'Year':          2025,
    'Month':         6,
}
if 'Mukim' in final_features:
    sample_input['Mukim'] = 'Pulai'   # most common Mukim in training set

sample_df = pd.DataFrame([sample_input])[final_features]

log_pred    = final_pipeline.predict(sample_df)[0]
central_rm  = np.expm1(log_pred)

print("=" * 55)
print("SAMPLE PREDICTION — Local Explanation")
print("=" * 55)
print(f"Input property:")
for k, v in sample_input.items():
    print(f"  {k:<22}: {v}")
print(f"\nPredicted log1p(Price) : {log_pred:.4f}")
print(f"Central estimate       : RM {central_rm:,.0f}")

# Explanation: show top 10 individual feature importances for context
print("\n--- Global feature importance context (top 10) ---")
print(imp_df.head(10)[['feature', 'importance']].to_string(index=False))

# SHAP local explanation if available
if HAS_SHAP and explainer is not None:
    X_sample_prep = prep.transform(sample_df)
    sv_sample     = explainer.shap_values(X_sample_prep)[0]
    contrib = pd.Series(sv_sample, index=feat_names).sort_values(key=np.abs, ascending=False)
    print("\n--- SHAP: Top 10 contributors for this prediction ---")
    print(f"  Base value (mean log prediction): {explainer.expected_value:.4f}")
    print(f"  Final prediction (log)           : {log_pred:.4f}")
    print()
    top_shap = contrib.head(10)
    for feat, val in top_shap.items():
        direction = '▲' if val > 0 else '▼'
        print(f"  {direction} {val:+.4f}  {feat}")

    try:
        shap.waterfall_plot(shap.Explanation(
            values=sv_sample,
            base_values=float(explainer.expected_value),
            data=X_sample_prep[0],
            feature_names=list(feat_names)
        ), max_display=12, show=False)
        plt.title('SHAP Waterfall — Sample Prediction')
        plt.tight_layout()
        plt.show()
    except Exception as e:
        print(f"Waterfall plot failed: {e}")
else:
    print("\n--- SHAP not available — using segment-based context ---")
    prop_type_median = seg.loc[seg['Property Type'] == sample_input['Property Type'],
                                'y_true_rm'].median()
    district_median  = seg.loc[seg['District'] == sample_input['District'],
                                'y_true_rm'].median()
    overall_median   = seg['y_true_rm'].median()
    print(f"  Overall test median price          : RM {overall_median:,.0f}")
    print(f"  Property type median (test set)    : RM {prop_type_median:,.0f}")
    print(f"  District median (test set)         : RM {district_median:,.0f}")
    print(f"  Model prediction                   : RM {central_rm:,.0f}")

## 13. Valuation Range

An AVM should not return a single point estimate. A residual-based empirical range is computed from the test set and used as an indicative upper/lower bound.

This is **not a statistical confidence interval**. It is an 80% empirical range: 80% of 2026 test-set predictions fell within these bounds.

In [ ]:
test_resid_log = y_test.values - final_pipeline.predict(final_X_test)

p10_log = float(np.percentile(test_resid_log, 10))
p90_log = float(np.percentile(test_resid_log, 90))

VALUATION_BOUNDS = {
    'lower_factor': float(np.exp(p10_log)),
    'upper_factor': float(np.exp(p90_log)),
    'p10_log': p10_log,
    'p90_log': p90_log,
    'description': (
        '80% empirical range from 2025–2026 test set residuals. '
        'Indicative only — not a statistical confidence interval.'
    )
}

print("=" * 60)
print("VALUATION BOUNDS (from test-set residuals)")
print("=" * 60)
print(f"  10th percentile residual (log) : {p10_log:+.4f}")
print(f"  90th percentile residual (log) : {p90_log:+.4f}")
print(f"  Lower bound factor             : {VALUATION_BOUNDS['lower_factor']:.4f}  "
      f"(≈ {(VALUATION_BOUNDS['lower_factor']-1)*100:+.1f}% of central estimate)")
print(f"  Upper bound factor             : {VALUATION_BOUNDS['upper_factor']:.4f}  "
      f"(≈ {(VALUATION_BOUNDS['upper_factor']-1)*100:+.1f}% of central estimate)")
print(f"\n  Example on RM 400,000 property:")
ex = 400_000
print(f"    Lower : RM {ex * VALUATION_BOUNDS['lower_factor']:,.0f}")
print(f"    Central: RM {ex:,.0f}")
print(f"    Upper : RM {ex * VALUATION_BOUNDS['upper_factor']:,.0f}")
print(f"\n  Description: {VALUATION_BOUNDS['description']}")

# Residual distribution plot
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(test_resid_log, bins=80, color='steelblue', edgecolor='none', density=True, alpha=0.7)
ax.axvline(p10_log, color='orange', lw=1.5, ls='--', label=f'P10 = {p10_log:+.3f}')
ax.axvline(p90_log, color='red',    lw=1.5, ls='--', label=f'P90 = {p90_log:+.3f}')
ax.axvline(0, color='black', lw=1)
ax.set_xlabel('Residual (actual − predicted, log-space)')
ax.set_title('Test-Set Residuals — 80% Empirical Range')
ax.legend()
plt.tight_layout()
plt.show()

## 14. Sample Prediction Function

A demonstration function that accepts property inputs and returns a structured valuation with range. This is a demonstration only — not an API endpoint.

In [ ]:
def predict_property_value(property_type, district, tenure,
                            land_area_sqm, main_floor_area_sqm=None,
                            year=2025, month=6,
                            mukim=None,
                            pipeline=None, features=None, bounds=None):
    """
    Demonstration prediction function.
    Returns: dict with central_estimate_rm, lower_bound_rm, upper_bound_rm.
    Handles missing Area (imputed by pipeline) and unseen categories.
    """
    if pipeline is None: pipeline = final_pipeline
    if features is None: features = final_features
    if bounds   is None: bounds   = VALUATION_BOUNDS

    area_val = float(main_floor_area_sqm) if main_floor_area_sqm is not None else np.nan

    row = {
        'Property Type': property_type,
        'Tenure':        tenure,
        'District':      district,
        'Land':          float(land_area_sqm),
        'Area':          area_val,
        'Year':          int(year),
        'Month':         int(month),
    }
    if 'Mukim' in features:
        row['Mukim'] = mukim if mukim else 'Unknown'

    input_df  = pd.DataFrame([row])[features]
    log_pred  = pipeline.predict(input_df)[0]
    central   = float(np.expm1(log_pred))
    lower     = central * bounds['lower_factor']
    upper     = central * bounds['upper_factor']

    return {
        'central_estimate_rm': round(central, -3),
        'lower_bound_rm':      round(lower,   -3),
        'upper_bound_rm':      round(upper,   -3),
        'range_note':          bounds['description'],
    }


# ── Demo calls ────────────────────────────────────────────────────────────────
test_cases = [
    dict(property_type='2 - 2 1/2 Storey Terraced', district='Johor Bahru',
         tenure='Freehold', land_area_sqm=150, main_floor_area_sqm=110,
         year=2025, month=6, mukim='Pulai'),

    dict(property_type='Condominium/Apartment', district='Kuala Lumpur',
         tenure='Leasehold', land_area_sqm=0, main_floor_area_sqm=None,
         year=2025, month=3, mukim=None),

    dict(property_type='1 - 1 1/2 Storey Terraced', district='Petaling',
         tenure='Freehold', land_area_sqm=120, main_floor_area_sqm=80,
         year=2025, month=1, mukim=None),
]

print("=" * 65)
print("DEMO — predict_property_value()")
print("=" * 65)
for i, tc in enumerate(test_cases, 1):
    result = predict_property_value(**tc)
    print(f"\nCase {i}: {tc['property_type']} | {tc['district']} | {tc['tenure']}")
    print(f"  Central estimate : RM {result['central_estimate_rm']:>10,.0f}")
    print(f"  Lower bound      : RM {result['lower_bound_rm']:>10,.0f}")
    print(f"  Upper bound      : RM {result['upper_bound_rm']:>10,.0f}")
    print(f"  Note: {result['range_note'][:60]}...")

In [ ]:
# ── Print final summary statistics for the Markdown cell below ───────────────
print("=" * 70)
print("NUMBERS FOR FINAL SUMMARY")
print("=" * 70)
print(f"  Dataset      : 416,627 rows × 13 columns, 2021–2026")
print(f"  Train rows   : {len(X_train_tune):,}  (2021–2024)")
print(f"  Test rows    : {len(final_X_test):,}  (2025–2026)")
print(f"  Final features: {final_features}")
print()
print(f"  Baseline Model A (no Mukim):")
print(f"    Train R² (log) = {tr_A['r2_log']:.4f}  |  Test R² (log) = {te_A['r2_log']:.4f}")
print(f"    Test RMSE = RM {te_A['rmse']:,.0f}  |  Test MAE = RM {te_A['mae']:,.0f}  |  Test MedAE = RM {te_A['medae']:,.0f}")
print()
print(f"  Best pre-tuning (Model {best_model_label}):")
print(f"    Train R² (log) = {best_te[0]['r2_log']:.4f}  |  Test R² (log) = {best_te[0]['r2_log']:.4f}")
print()
print(f"  Tuned Model {best_model_label}:")
print(f"    Train R² (log) = {tr_tuned['r2_log']:.4f}  |  Test R² (log) = {te_tuned['r2_log']:.4f}")
print(f"    Test RMSE = RM {te_tuned['rmse']:,.0f}  |  Test MAE = RM {te_tuned['mae']:,.0f}  |  Test MedAE = RM {te_tuned['medae']:,.0f}")
print(f"    RMSE % median = {te_tuned['rmse_pct']:.1f}%")
print()
print(f"  Best params: {rf_tuned_params}")
print()
print(f"  Valuation range factors: lower={VALUATION_BOUNDS['lower_factor']:.4f}, upper={VALUATION_BOUNDS['upper_factor']:.4f}")
print()
print(f"  Top feature groups (impurity importance):")
for _, row_fi in group_imp_df.head(5).iterrows():
    print(f"    {row_fi['Feature Group']:<20}: {row_fi['Importance']:.4f}")

## 17. Final Summary

### 1. Dataset Used
416,627 residential property transactions across Malaysia, 2021–2026 (13 columns). Source: `Open Transaction Data Cleaned.xlsx`. Key facts: `Price` skewness = 9.786; `Area` 25.9% missing with 100% missingness for all strata types (Condominium/Apartment, Flat, Low-Cost Flat, Town House).

### 2. Final Feature Set
Eight features, selected after a three-way Mukim encoding comparison (Models A/B/C). **Model C** (Mukim with TargetEncoder) was chosen: Test R² = 0.8589 vs Model A 0.8430, Model B 0.8572.

| Feature | Encoding |
|---|---|
| Property Type (11 cat.) | OneHotEncoder |
| Tenure (2 cat.) | OneHotEncoder |
| District (127 cat.) | OneHotEncoder |
| Mukim (1,343 cat.) | TargetEncoder(smooth='auto') |
| Land | SimpleImputer(median) → log1p |
| Area | PropertyTypeMedianImputer → log1p |
| Year | Passthrough |
| Month | Passthrough |

`Road Name` (121,825 unique + 25.9% blank) and `Scheme Name/Area` (23,718 unique) excluded.

### 3. Preprocessing Strategy
Target: `y = log1p(Price)`. Back-transform: `expm1`. All preprocessing inside a `Pipeline + ColumnTransformer` — no test-set leakage. `PropertyTypeMedianImputer` fills strata Area with per-type training median. Mukim `fillna('Unknown').astype(str)` applied before TargetEncoder.

### 4. Train/Test Split
Chronological: **Train = 410,959 rows (2021–2025)**, **Test = 5,668 rows (2026 — January to March only)**. The 2026 year is reserved entirely as a held-out evaluation set to simulate deployment conditions. Note: the small test set (1.4% of data, 3 months) makes metrics more volatile than a larger holdout.

### 5. Baseline RF Result (Model A — no Mukim, 50 trees)

| Split | R² (log) | RMSE | MAE | Median AE | RMSE % median |
|---|---|---|---|---|---|
| Train | 0.9817 | RM 100,529 | RM 32,234 | RM 14,437 | 26.4% |
| Test (2026) | 0.8430 | RM 243,445 | RM 100,551 | RM 47,780 | 64.1% |
| OOB | 0.8948 | — | — | — | — |

OOB R² = 0.8948 indicates strong training-set generalisation. Test R² = 0.8430 on 2026 data coincidentally matches the OLS benchmark (0.843), suggesting both models capture a similar proportion of price variance. Train/test gap (0.14) indicates overfitting.

### 6. Tuned RF Result (Model C — Mukim target encoding, 200 trees)

| Split | R² (log) | RMSE | MAE | Median AE | RMSE % median |
|---|---|---|---|---|---|
| Train | 0.9732 | — | — | — | — |
| **Test (2026)** | **0.8572** | **RM 225,105** | **RM 92,369** | **RM 44,831** | **59.2%** |

Best hyperparameters (RandomizedSearchCV, TimeSeriesSplit n=5, 80K chronological sample, 15 iterations): `max_depth=None`, `max_features=0.5`, `min_samples_leaf=1`, `min_samples_split=2`, `n_estimators=200`. Tuning gave modest gain over the pre-tuning Model C baseline in RMSE/MAE (RM 228K → RM 225K); test R² is near-identical (0.8589 → 0.8572) within the noise of a small 5,668-row test set.

### 7. Comparison Against OLS Benchmark

| Model | R² (log) | Split | Rows | Test period |
|---|---|---|---|---|
| OLS | 0.843 | None (full dataset) | ∼304K (outliers removed) | — |
| RF Baseline (Model A) | 0.843 | Chronological | 411K train / 5.7K test | 2026 (Jan–Mar) |
| RF Tuned (Model C) | **0.857** | Chronological | 411K train / 5.7K test | 2026 (Jan–Mar) |

The tuned RF improves on OLS (R² 0.857 vs 0.843) under a stricter chronological holdout. However, the test set is small (5,668 rows, 3 months) — these numbers should be interpreted cautiously. With more 2026 data, the estimate would be more stable.

### 8. Segment-Level Performance (2026 test set)

**By Property Type** — Weakest: Low-Cost Flat 18.7%, Detached 16.6%, Flat 16.4%, Low-Cost House 16.0%. Best: Cluster House 9.0%, 1–1.5 Storey Semi-Detached 10.1%.

**By Price Band** — >RM 1M: 18.1% (worst), ≤RM 300K: 13.5%, RM 500K–1M: 12.7%, RM 300K–500K: 10.6% (best). The core market (RM 300K–500K) remains the most accurately valued.

**Districts (top 15 by 2026 volume)** — Weakest: Kuala Lumpur 16.3%, Timur Laut (Penang) 15.3%, Johor Bahru 13.9%.

**Note**: Small test set means each segment has fewer observations — segment metrics are more sensitive to individual high-error predictions than with larger test sets.

### 9. Feature Importance Findings

| Feature Group | Impurity Importance | Permutation Importance (R² drop) |
|---|---|---|
| Area | 0.325 (1st) | 0.237 (3rd) |
| Land | 0.226 (2nd) | **0.553 (1st)** |
| Mukim | 0.198 (3rd) | 0.312 (2nd) |
| Property Type | 0.101 (4th) | 0.188 (4th) |
| District | 0.088 (5th) | 0.136 (5th) |
| Month | 0.016 | ≈0 |
| Year | 0.013 | 0.000 |
| Tenure | 0.012 | 0.034 |

Land is #1 by permutation despite ranking 2nd by impurity — consistent with previous findings (impurity bias toward Area due to NaN-imputed strata values). Mukim is a strong sub-district signal (0.31 R² drop). Year permutation importance = 0 on the 2026 test set: the model does not use time as a predictor, which is consistent with the chronological split design but represents a deployment risk if 2027+ prices deviate significantly.

### 10. Model Weaknesses

1. **Small 2026 test set**: Only 5,668 rows (Jan–Mar 2026, 1.4% of data). Metrics are more volatile; segment-level results are based on small samples. Results will stabilise as more 2026 data accumulates.
2. **Overfitting**: Train R² 0.973 vs Test R² 0.857 despite tuning.
3. **Strata Area gap**: 100% of strata type rows have imputed Area. Cannot distinguish unit sizes within a Mukim.
4. **Luxury volatility**: >RM 1M properties: 18.1% Median % Error, RMSE ≈ RM 768K. Sparse training data + high variance from non-observable features.
5. **Temporal sensitivity**: Year permutation importance = 0 — the model does not explicitly model 2026 price levels. Systematic under/over-prediction if 2026 prices deviate from 2021–2025 patterns.
6. **Affordable housing**: Low-Cost Flat (18.7%) and Low-Cost House (16.0%) errors are highest — government pricing constraints not captured.

### 11. AVM Suitability Assessment

The model is **suitable for indicative AVM purposes in the core Malaysian residential market (RM 300K–RM 1M)**, achieving Median % Error of 10.6–12.7% and Test R² = 0.857 on the 2026 chronological holdout. The RF outperforms the OLS benchmark under a stricter evaluation protocol.

**Caveat**: The 2026 test set is small (5,668 rows, 3 months) — these estimates will become more robust as the year progresses and more transaction data is available.

**Not recommended** as the sole basis for: luxury (>RM 1M), government/subsidised housing, new launches (no Mukim encoding history), or certified appraisals.

The 80% empirical valuation range (−21.5% lower, +33.1% upper) is asymmetric, reflecting the right-skewed residual distribution on the 2026 test set.

### 12. Next Modelling Step

1. **Re-evaluate as more 2026 data arrives**: The 5,668-row test set covers Jan–Mar only. Re-running this notebook on a full-year 2026 test set will give stable segment-level and overall metrics.
2. **Scheme Name/Area (Model D)**: Frequency-encode 23,718 scheme names — the most immediately actionable feature improvement.
3. **Gradient Boosting**: XGBoost or LightGBM — expected to further improve over RF's R² 0.857.
4. **Strata Area recovery**: Source actual unit sizes from NAPIC — the single largest data improvement opportunity.
5. **Temporal price trend features**: Rolling 12-month District/Mukim median price to address near-zero Year importance and improve robustness to market drift.

## 18. Limitations

1. **Strata Area unavailable**: The NAPIC transaction dataset does not record main floor area for strata properties. All ∼67,000 strata training rows received a type-median Area imputation. The model cannot price unit-size variation within a strata development.

2. **No building-level identifiers**: Without a scheme-level identifier (building name, strata title), the model cannot learn building-specific premiums or discounts. Two condominiums in the same Mukim can differ by 50% based on age and facilities — not captured here.

3. **Temporal features are weak**: Year and Month carry near-zero permutation importance. The model treats 2025 and 2021 prices as essentially the same conditional on location and property attributes. In a rising or falling market, this produces systematic over- or under-valuation.

4. **OLS comparison is indicative only**: OLS R² = 0.843 used ∼304K rows, no train/test split, and outliers removed at ±3σ. RF Test R² = 0.858 used a stricter out-of-time holdout. These figures are not directly comparable.

5. **Luxury and rare property types**: Detached (2,597 test rows), Cluster Houses (629), and Town Houses (321) are underrepresented. RF cannot extrapolate beyond observed price ranges; unique high-end properties may be unreliably valued.

6. **Geographic coverage**: Districts with few transactions (rural Sabah/Sarawak) may have unreliable predictions due to limited Mukim-level target encoding training signal.

7. **Unseen Mukim**: TargetEncoder assigns the training grand mean to Mukim values unseen during training (new developments). These predictions are less accurate than those for established Mukim with rich training history.

8. **No macroeconomic features**: Interest rates, economic growth, and housing policy changes are not included. The model cannot anticipate the impact of a rate change on future prices.

## 19. Next Steps

### Immediate (next modelling iteration)
1. **Model D — Scheme Name/Area ablation**: Frequency-encode `Scheme Name/Area` (23,718 unique values; rare schemes → “Other”) and compare test R² against Model C. Expected improvement: 1–3 R² points for strata properties within known schemes.
2. **Unit floor area for strata**: Obtain actual built-up area from NAPIC or developer brochures. Even partial coverage would significantly improve strata property accuracy.

### Near-term improvements
3. **Gradient Boosting comparison**: Train XGBoost or LightGBM with the same chronological split and hyperparameter search. Gradient boosting handles mixed feature types and continuous interactions more efficiently than RF.
4. **Temporal feature engineering**: Compute rolling 12-month median price per District and per Mukim as additional features to encode local price trends and address the near-zero Year/Month importance.
5. **Calibrated prediction intervals**: Replace the empirical P10/P90 range with quantile regression forests or conformal prediction for tighter, better-calibrated intervals.

### Longer-term
6. **Address-level geospatial features**: Add proximity to nearest MRT/LRT, CBD, highway, and schools using OpenStreetMap or official GIS layers. This would reduce the Mukim-level averaging effect.
7. **Property cycle regime indicator**: Build a separate model (HMM or threshold-based) classifying the market as expansion/plateau/contraction. Feed the regime label as a feature to improve temporal robustness.
8. **Monitoring and retraining**: If deployed, monitor prediction drift and residual distribution over time. Retrain quarterly on rolling 4-year windows as new NAPIC transaction data becomes available.